# Анализ эффективности новой версии сайта: A/B-тест по целевой метрике «конверсия в покупку».

- Автор: Стукалов Артем Витальевич
- Дата: 15.03.2026

## 1. Цели исследования.

#### Цель проекта:
Оценить эффективность новой версии сайта интернет-магазина товаров для здорового образа жизни, направленной на упрощение интерфейса, и определить, достигается ли целевое увеличение конверсии зарегистрированных пользователей в покупателей на 3 процентных пункта в течение первых семи дней после регистрации.

**Особенности А/В теста:**
- **Гипотеза:** Упрощение интерфейса сайта приведёт к тому, что в течение семи дней после регистрации конверсия зарегистрированных пользователей в покупателей увеличится как минимум на 3 процентных пункта по сравнению с контрольной группой (старый дизайн).
- **Целевая метрика:** Конверсия в покупку в течение 7 дней после регистрации.

#### Задачи проекта:
- **Загрузка данных и оценка их целостности:**
    - Загрузить данные датасетов: ab_test_participants и ab_test_events.
    - Провести предобработку данных датасетов на предмет наличия пропусков и дубликатов.
- **По таблице ab_test_participants оценить корректности проведения теста:**
    - Отфильтровать данные датасета ab_test_participants и выбирать только те строки, которые относятся к тесту interface_eu_test.
    - Исключить пересечения с конкурирующим тестом.
    - Проверить равномерность распределения пользователей по группам теста.
- **Проанализировать данные о пользовательской активности по таблице ab_test_events:**
    - Оставить только события, связанные с участвующими в изучаемом тесте пользователями.
    - Определить горизонт анализа: рассчитать время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации.
    - Оценить достаточность выборки для получения статистически значимых результатов A/B-теста.
    - Рассчитать для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.
    - Сделать предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.
- **Провести оценку результатов A/B-тестирования:**
    - Проверить изменение конверсии подходящим статистическим тестом.
    - Составить выводы по проведённой оценке результатов A/B-тестирования.

#### Описание данных:
В рамках тестирования новой версии сайта собраны два связанных набора данных:
1. **Таблица таблица участников тестов:**
    - user_id — идентификатор пользователя.
    - group — группа участия в тесте: A (контрольная, старый дизайн) или B (экспериментальная, новый интерфейс);
    - ab_test — название теста;
    - device — устройство, с которого происходила регистрация.
2. **Таблица событий пользователей:**
    - user_id — идентификатор пользователя;
    - event_dt — дата и время события;;
    - event_name — тип события;
    - details — дополнительные данные о событии.

## 2. Загрузка данных и оценка их целостности.

Импортируем основные библиотеки.

In [1]:
import pandas as pd # Импортируем библиотеку pandas

# Загружаем библиотеки для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

Загрузим данные датасетов ab_test_participants и ab_test_events.

In [2]:
# Выгружаем данные в переменные participants и events
participants = pd.read_csv('ab_test_participants.csv')
events = pd.read_csv('ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

**В начале изучим данные датасета participants.**

In [3]:
participants.head()  # выводим первые строки датасета participants

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [4]:
participants.info() # выводим основную информацию датасета participants

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


Проверим данные датасета participants на их целостность.

Узнаем о распределении количества пропусков между столбцами датасета participants.

In [5]:
participants.isna().sum() # Считаем количество пропусков по столбцам

user_id    0
group      0
ab_test    0
device     0
dtype: int64

Как видно из результата, в датасете participants отсутствуют пропуски.

Проверим данные на наличие явных (полных) дубликатов в датасете participants.

In [6]:
duplicates_participants_count = participants.duplicated().sum() # считаем количество полных дубликатов
print(f"Количество дубликатов в датасете participants: {duplicates_participants_count}")

Количество дубликатов в датасете participants: 0


Проверим данные на наличие явных (полных) дубликатов в идентификаторах пользователей датасета participants.

In [7]:
duplicates_participants = participants.duplicated(subset=['user_id']).sum() # считаем количество дубликатов по user_id
print(f"Количество дубликатов в user_id датасета participants: {duplicates_participants}")

Количество дубликатов в user_id датасета participants: 887


Несмотря на наличие дубликатов по user_id в датасете participants мы не можем их удалить, поскольку  один и тот же пользователь может участвовать в нескольких разных тестах.

**Изучим данные датасета events.**

In [8]:
events.head() # выводим первые строки датасета events

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [9]:
events.info() # выводим основную информацию датасета events

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


Проверим данные датасета events на их целостность.

Узнаем о распределении количества пропусков между столбцами датасета participants.

In [10]:
events.isna().sum() # Считаем количество пропусков по столбцам

user_id            0
event_dt           0
event_name         0
details       538264
dtype: int64

In [11]:
events.isna().mean() * 100 # Считаем количество пропусков по столбцам в процентах

user_id        0.000000
event_dt       0.000000
event_name     0.000000
details       68.369563
dtype: float64

Из результата видно, что в столбце details датасета events обнаружено 68% пропусков.  Столбец details содержит дополнительные данные, удаление таких пропусков в столбце details лишает важнейшей информации о действиях пользователей, необходимой для дальнейшего анализа. Таким образом, столбец details следует оставить без изменения.

Проверим данные на наличие явных (полных) дубликатов в идентификаторах пользователей датасета events.

In [12]:
duplicates_events = events.duplicated(subset=['user_id']).sum() # считаем количество дубликатов по user_id
print(f"Количество дубликатов в user_id : {duplicates_events}")

Количество дубликатов в user_id : 643102


В данном случае duplicated(subset=['user_id']) считает количество повторяющихся строк, у которых одинаковый user_id. Но в таблице событий каждый пользователь совершает множество действий, поэтому у него будет много строк. Это не дубликаты, а детальная история активности, поэтому мы не можем их удалить, поскольку это уничтожит все данные о поведении пользователей.

Проверим данные на наличие явных (полных) дубликатов в датасете events.

In [13]:
duplicates_events_count = events.duplicated().sum() # считаем количество дубликатов
print(f"Количество дубликатов датасете events: {duplicates_events_count}")

Количество дубликатов датасете events: 36318


В результате обнаружено 36318 полных дубликатов строк в датасете events. Это значит, что существуют строки, где все значения столбцов (user_id, event_dt, event_name, details) полностью совпадают.
Между тем в событийной аналитике каждая запись фиксирует действие пользователя, и повторяющиеся события могут быть частью реального поведения. Удаляя такие записи, можно исказить картину пользовательского пути и повлиять на расчёт метрик. Таким образом, мы оставляем данные без изменений.

**Промежуточный вывод:**
- **Датасет participants:**
    - В датасете participants пропусков нет, все столбцы заполнены.
    - В датасете participants найдено 887 дубликатов по user_id, возможно это связано с участием одних и тех же пользователей в разных тестах. Их не следует удалять, поскольку это приведёт к потере информации. 
- **Датасет events:**
    - В столбце details датасета events выявлено 68% пропусков.  Данный столбец оставлен без изменений, поскольку это дополнительные данные и удаление в них пропусков лишило бы  информации о действиях пользователей.
    - По user_id датасета events найдены дубликаты в количестве 643 102. Между тем один пользователь может совершать множество событий, а значит данные дубликаты по идентификатору пользователя не следует удалять.
    - В датасете events обнаружено 36 318 полных дубликатов строк. Эти строки оставлены как есть, поскольку удаляя такие записи, можно исказить картину пользовательского пути и повлиять на расчёт метрик.

## 3. Оценка корректности проведения теста:

Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания

   - равномерность распределения пользователей по группам теста

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

**Требования технического задания:**
- Название теста: interface_eu_test.
- Группы: А (контрольная), B (новый интерфейс).
- Каждый пользователь должен принадлежать только одной группе в рамках этого теста.

In [14]:
participants.head()  # выводим первые строки датасета participants

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [15]:
# смотрим уникальные значения столбца ab_test
participants['ab_test'].unique()

array(['interface_eu_test', 'recommender_system_test'], dtype=object)

Как можно увидеть в столбце ab_test датасета participants находятся два разных теста: 'interface_eu_test' и 'recommender_system_test'.

Фильтруем данные и выбираем только те строки, которые относятся к тесту interface_eu_test.

In [16]:
# Отбираем только строки, относящиеся к тесту interface_eu_test.
interface_eu_test = participants[participants['ab_test'] == 'interface_eu_test'].copy()

Считаем количество строк теста interface_eu_test

In [17]:
print(f'Количество строк в датасете interface_eu_test: {len(interface_eu_test)}')

Количество строк в датасете interface_eu_test: 10850


Проверим равномерность распределения пользователей по группам теста.

In [18]:
# считаем количество уникальных пользователей в каждой из групп
group_counts = interface_eu_test['group'].value_counts().sort_values()
print("Распределение пользователей по группам:")
print(group_counts)

Распределение пользователей по группам:
group
A    5383
B    5467
Name: count, dtype: int64


In [19]:
# считаем процентное распределение уникальных пользователей в каждой из групп
group_counts_percentage = interface_eu_test['group'].value_counts(normalize=True)*100
print(f'Распределение пользователей по группам: {group_counts_percentage}')

Распределение пользователей по группам: group
B    50.387097
A    49.612903
Name: proportion, dtype: float64


Проверим отсутствие пересечений с конкурирующим тестом.

In [20]:
# Отбираем только строки, относящиеся к тесту recommender_system_test.
recommender_system_test = participants[participants['ab_test'] == 'recommender_system_test'].copy()

# Получаем множества уникальных пользователей
users_interface = set(interface_eu_test['user_id'].unique())
users_recommender = set(recommender_system_test['user_id'].unique())

# Проверим пересечения с конкурирующим тестом.
intersection_users = users_interface & users_recommender
print(f'Количество пользователей, пересекающихся с другими тестами: {len(intersection_users)}')

Количество пользователей, пересекающихся с другими тестами: 887


Обнаруженное пересечение в 887 пользователей означает, что эти люди одновременно участвовали в двух разных A/B тестах: interface_eu_test и recommender_system_test. Такое перекрытие нарушает чистоту эксперимента, так как на поведение этих пользователей могли повлиять оба изменения, и невозможно определить, какой именно фактор вызвал наблюдаемые эффекты. Поэтому для анализа результатов interface_eu_test пользователей с пересечнием в тестах необходимо исключить.

In [21]:
# Исключаем пользователей, которые есть в intersection_users
interface_eu_test_clean = interface_eu_test[interface_eu_test['user_id'].isin(intersection_users) == False].copy()

# Проверяем результат
diff = len(interface_eu_test) - len(interface_eu_test_clean)

print(f'Количество строк в interface_eu_test_clean: {len(interface_eu_test_clean)}')
print(f'Количество строк, которые были исключены: {diff}')

Количество строк в interface_eu_test_clean: 9963
Количество строк, которые были исключены: 887


Посмотрим как изменилось распределение пользователей по группам теста после исключения из данных пользователей, участвующих  в двух разных A/B тестах.

In [22]:
# Считаем количество уникальных пользователей в каждой из групп после исключения пользователей, участвующих в двух разных A/B тестах
group_counter = interface_eu_test_clean['group'].value_counts().sort_values()
print("Распределение пользователей по группам:")
print(group_counter)

Распределение пользователей по группам:
group
A    4952
B    5011
Name: count, dtype: int64


In [23]:
# Считаем новое процентное распределение
group_counts_percentage_clean = interface_eu_test_clean['group'].value_counts(normalize=True)*100
print(f'Распределение пользователей по группам: {group_counts_percentage_clean}')

Распределение пользователей по группам: group
B    50.296096
A    49.703904
Name: proportion, dtype: float64


**Промежуточный вывод:**
- Были отфильтрованы строки по названию теста interface_eu_test
- Была проведена проверка распределения пользователей теста interface_eu_test по группам, в ходе которой были получены следующие результаты:
    - Группа A: 4952 пользователей (50.29%)
    - Группа B: 5011 пользователей (49.70%)
- Обнаружено 887 пользователей, которые одновременно участвуют в тестах interface_eu_test и recommender_system_test. Это нарушает чистоту эксперимента, так как их поведение может быть искажено влиянием двух разных тестов.
- Из выборки interface_eu_test исключены все строки, соответствующие этим 887 пользователям. После очистки получен датасет interface_eu_test_clean, содержащий 9963 строки. Количество исключённых строк совпадает с числом пересекающихся пользователей, что подтверждает корректность фильтрации.

**Проанализируем данные о пользовательской активности по таблице `ab_test_events`.**

Оставим только события, связанные с участвующими в изучаемом тесте пользователями.

In [24]:
# Получаем список пользователей теста
unique_test_users = interface_eu_test_clean['user_id'].unique()

# Фильтруем события
events_test = events[events['user_id'].isin(unique_test_users)].copy()

print(f'Всего событий для участников interface_eu_test: {len(events_test)}')
print(f'Уникальных пользователей среди событий: {events_test["user_id"].nunique()}')

Всего событий для участников interface_eu_test: 73815
Уникальных пользователей среди событий: 9963


Определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации.

In [25]:
# Определяем минимальную дату события для каждого пользователя
user_start = events_test.groupby('user_id')['event_dt'].min().reset_index()
user_start.columns = ['user_id', 'start_date']

# Присоединяем start_date ко всем событиям
events_with_start = events_test.merge(user_start, on='user_id', how='inner').copy()

# Вычисляем разницу во времени (timedelta)
events_with_start['time_from_start'] = events_with_start['event_dt'] - events_with_start['start_date']

# Оставляем события, произошедшие не позднее 7 дней от start_date
mask_7days = events_with_start['time_from_start'] <= pd.Timedelta(days=7)
events_7days = events_with_start[mask_7days].copy()

print(f'Событий в первые 7 дней от первого действия пользователя: {len(events_7days)}')

Событий в первые 7 дней от первого действия пользователя: 63805


In [26]:
events_7days.sample(10)

,user_id,event_dt,event_name,details,start_date,time_from_start
22045,14DED88B574B52CB,2020-12-14 07:11:05,login,NaN,2020-12-12 21:30:47,1 days 09:40:18
65008,DFB81B03716068DE,2020-12-24 05:09:07,purchase,4.99,2020-12-19 19:33:01,4 days 09:36:06
33993,F091C2F29A8FFBBC,2020-12-17 07:57:04,registration,-4.06,2020-12-17 07:57:04,0 days 00:00:00
21976,6CA5AF221442A856,2020-12-14 06:52:44,login,NaN,2020-12-10 21:22:14,3 days 09:30:30
33960,E5BD47F80D41D126,2020-12-17 07:41:44,login,NaN,2020-12-17 07:40:31,0 days 00:01:13
37132,BEC73402D56C9E8B,2020-12-18 04:17:58,product_page,NaN,2020-12-17 15:25:28,0 days 12:52:30
16155,7B1498FD201F52FB,2020-12-12 22:51:45,login,NaN,2020-12-12 22:46:40,0 days 00:05:05
49216,0C32A0CF544C7941,2020-12-21 00:58:04,purchase,9.99,2020-12-15 12:15:00,5 days 12:43:04
67976,29AC71A95A562545,2020-12-26 08:19:33,product_cart,NaN,2020-12-23 18:40:42,2 days 13:38:51
17016,9B6717844DFDFB16,2020-12-13 03:44:26,registration,0.0,2020-12-13 03:44:26,0 days 00:00:00


Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [27]:
# Параметры
p = 0.30  # базовый показатель конверсии
mde = 0.03 # минимальный детектируемый эффект
alpha = 0.05 # уровень значимости
power = 0.8 # мощность теста
effect_size = proportion_effectsize(p, p + mde)

# Инициализируем класс NormalIndPower
power_analysis = NormalIndPower()

# Расчёт размера выборки
sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1 # Равномерное распределение выборок
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3761


**Рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.**

In [28]:
# Создаём датасет всех участников теста с их группами
all_users = interface_eu_test_clean[['user_id', 'group']]

# Определяем покупателей
buyers = events_7days[events_7days['event_name'] == 'purchase']['user_id'].unique()
buyers_df = pd.DataFrame({'user_id': buyers, 'buyer_flag': 1})

# Присоединяем флаг покупки к данным о пользователях
all_users_with_buyers = all_users.merge(buyers_df, on='user_id', how='left')
all_users_with_buyers['buyer_flag'] = all_users_with_buyers['buyer_flag'].fillna(0).astype(int)

# Считаем общее число пользователей и количество покупателей
result_df = all_users_with_buyers.groupby('group').agg(
    total_visitors=('user_id', 'count'),
    buyers_count=('buyer_flag', 'sum')
).reset_index()

print(result_df)

  group  total_visitors  buyers_count
0     A            4952          1377
1     B            5011          1480


In [29]:
# Посчитаем конверсию в процентах
result_df['conversion_rate'] = (result_df['buyers_count'] / result_df['total_visitors'] * 100).round(2)

print(result_df)

  group  total_visitors  buyers_count  conversion_rate
0     A            4952          1377            27.81
1     B            5011          1480            29.54


**Предварительный вывод:**
- В тестовой группе B (новый упрощённый интерфейс) наблюдается более высокая конверсия в покупку по сравнению с контрольной группой A:
29,54% против 27,81%, абсолютный прирост составляет 1,73 процентных пункта.
- Однако для окончательного заключения необходимо провести статистическую проверку, чтобы убедиться, что различие не вызвано случайностью. Тем не менее предварительно можно говорить о положительной динамике в группе с новым интерфейсом.

## 4. Оценка результатов A/B-тестирования:

Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

**Особенности А/В теста:**
- **Гипотеза:** Упрощение интерфейса сайта приведёт к тому, что в течение семи дней после регистрации конверсия зарегистрированных пользователей в покупателей увеличится как минимум на три процентных пункта по сравнению с контрольной группой (старый дизайн).
- **Нулевая гипотеза (H₀):** Уровень конверсии останется неизменным.
- **Альтернативная гипотеза (H₁):** Конверсия зарегистрированных пользователей в покупателей увеличится и это увеличение будет статистически значимым.

Поскольку нам необходимо определить, существует ли значимая разница между долями (пропорциями) в двух независимых выборках, в таком случае целесообразно применить Z-тест пропорций.

In [30]:
# Извлекаем данные для групп A и B
n_a = result_df[result_df['group'] == 'A']['total_visitors'].values[0]
n_b = result_df[result_df['group'] == 'B']['total_visitors'].values[0]
m_a = result_df[result_df['group'] == 'A']['buyers_count'].values[0]
m_b = result_df[result_df['group'] == 'B']['buyers_count'].values[0]

# Доли конверсии
p_a = m_a / n_a
p_b = m_b / n_b

print(f"Группа A: покупателей {m_a} из {n_a} (конверсия {p_a:.2%})")
print(f"Группа B: покупателей {m_b} из {n_b} (конверсия {p_b:.2%})")
print(f"Разница в значениях конверсий (B - A): {p_b - p_a:.4f} ({ (p_b - p_a)*100:.2f} процентных пункта.)")

# Проверка предпосылки о достаточном количестве данных
if (m_a >= 5) and (n_a - m_a >= 5) and (m_b >= 5) and (n_b - m_b >= 5):
    print('Предпосылка о достаточном количестве данных выполняется!')
else:
    print('Предпосылка о достаточном количестве данных НЕ выполняется!')

alpha = 0.05  # уровень значимости

# Проверяем увеличится ли конверсия
stat_ztest, p_value_ztest = proportions_ztest(
    [m_a, m_b],
    [n_a, n_b],
    alternative='smaller'
)

print(f'pvalue={p_value_ztest}')

if p_value_ztest > alpha:
    print('Нулевая гипотеза находит подтверждение!')
else:
    print('Нулевая гипотеза не находит подтверждения!')

text_interpretation = 'увеличится' if p_value_ztest <= alpha else 'не изменится'

print(f'При упрощение интерфейса конверсия зарегистрированных пользователей в покупателей {text_interpretation}')

Группа A: покупателей 1377 из 4952 (конверсия 27.81%)
Группа B: покупателей 1480 из 5011 (конверсия 29.54%)
Разница в значениях конверсий (B - A): 0.0173 (1.73 процентных пункта.)
Предпосылка о достаточном количестве данных выполняется!
pvalue=0.028262547212292124
Нулевая гипотеза не находит подтверждения!
При упрощение интерфейса конверсия зарегистрированных пользователей в покупателей увеличится


**На основе проведенного A/B-тестирования можно сделать следующие выводы:**
- **Наблюдаемый эффект:**
    - Конверсия в контрольной группе (A) составила 27,81% (1377 покупок из 4952 пользователей),
    - Конверсия в тестовой группе (B) составила 29,54% (1480 из 5011).
    - Разница в значениях конверсий 1.73 процентных пункта в пользу нового интерфейса.
- **Статистическая проверка:**
    - Во время проведения Z-тест пропорций был получен p-value = 0,0282, что ниже уровня значимости 0,05. Это означает, что наблюдаемое различие статистически значимо: новый интерфейс приводит к увеличению конверсии в покупку по сравнению со старым.
- **Ожидаемый эффект:**
    - В исходной постановке задачи (техническое задание) требовалось, чтобы увеличение конверсии составило минимум 3 процентных пункта. Фактический прирост (1,73 роцентных пункта) оказался ниже этого порога. Таким образом, целевой эффект в 3 процентных пункта не достигнут.
    - Несмотря на то что целевой эффект в 3 процентных пункта не достигнут, в рамках текущей формулировки гипотезы (где проверялось просто наличие увеличения конверсии в покупку) можно заключить, что эффект улучшения конверсии подтверждён.